In [1]:
import pandas as pd

In [10]:
sumStats = pd.read_fwf('/scratch/eleiterw/AppliedProject/results/response_betterAge_oldEigen_complete.assoc.logistic')
print(sumStats.head())

   CHR                       SNP      BP A1 TEST NMISS      OR      SE  \
0    1  1:769257:A:G:rs142559957  769257  A  ADD   499  0.4243  0.3847   
1    1  1:773755:C:T:rs528768073  773755  C  ADD   499   7.158   1.063   
2    1    1:794299:C:G:rs4951859  794299  C  ADD   499  0.9513    0.18   
3    1  1:796073:A:G:rs186002080  796073  A  ADD   499  0.5384  0.5964   
4    1  1:797392:A:G:rs541848210  797392  A  ADD   499  0.7499  0.6438   

      L95     U95     STAT        P  
0  0.1996  0.9018   -2.229  0.02583  
1  0.8903   57.54    1.851  0.06421  
2  0.6685   1.354  -0.2776   0.7813  
3  0.1673   1.733   -1.038   0.2992  
4  0.2123   2.649   -0.447   0.6549  


In [11]:
sumStats['A2'] = sumStats['SNP'].str.split(':', expand = True)[3]

In [12]:
print(sumStats.head())

   CHR                       SNP      BP A1 TEST NMISS      OR      SE  \
0    1  1:769257:A:G:rs142559957  769257  A  ADD   499  0.4243  0.3847   
1    1  1:773755:C:T:rs528768073  773755  C  ADD   499   7.158   1.063   
2    1    1:794299:C:G:rs4951859  794299  C  ADD   499  0.9513    0.18   
3    1  1:796073:A:G:rs186002080  796073  A  ADD   499  0.5384  0.5964   
4    1  1:797392:A:G:rs541848210  797392  A  ADD   499  0.7499  0.6438   

      L95     U95     STAT        P A2  
0  0.1996  0.9018   -2.229  0.02583  G  
1  0.8903   57.54    1.851  0.06421  T  
2  0.6685   1.354  -0.2776   0.7813  G  
3  0.1673   1.733   -1.038   0.2992  G  
4  0.2123   2.649   -0.447   0.6549  G  


In [5]:
imputationStats = pd.read_table('/scratch/eleiterw/AppliedProject/annotatedBCF/imputationStatistics0.txt', sep = '|', low_memory=False)
print(imputationStats.head())

  CHROM    POS            ID REF ALT  IMPUTED  TYPED       MAF        R2
0    10  47029   rs754551324   T  TC        1      0  0.001891  0.913743
1    10  47196  rs1456332226   G   A        1      0  0.000872  0.922933
2    10  47588   rs200635479   G   A        1      0  0.032330  0.810721
3    10  47711  rs1488057130   G   A        1      0  0.000398  0.420770
4    10  48486    rs10904045   C   T        1      0  0.396465  0.787442


In [13]:
# Convert chromosome columns to the same type (string recommended)
sumStats["CHR"] = sumStats["CHR"].astype(str)
imputationStats["CHROM"] = imputationStats["CHROM"].astype(str)

# Convert position columns to the same type (integer recommended)
sumStats["BP"] = sumStats["BP"].astype(int)
imputationStats["POS"] = imputationStats["POS"].astype(int)

In [14]:
merged_df_A1 = sumStats.merge(imputationStats, how="left", left_on=["CHR", "BP", "A1"], right_on=["CHROM", "POS", "REF"])
merged_df_A2 = sumStats.merge(imputationStats, how="left", left_on=["CHR", "BP", "A2"], right_on=["CHROM", "POS", "REF"])

In [15]:
merged_df = pd.concat([merged_df_A2, merged_df_A1])
merged_df = merged_df.drop_duplicates(subset=["CHR", "BP", "A1", "A2", "REF", "ALT"])
merged_df = merged_df.dropna()
print(merged_df.head())

  CHR                       SNP      BP A1 TEST NMISS      OR      SE     L95  \
0   1  1:769257:A:G:rs142559957  769257  A  ADD   499  0.4243  0.3847  0.1996   
1   1  1:773755:C:T:rs528768073  773755  C  ADD   499   7.158   1.063  0.8903   
3   1  1:796073:A:G:rs186002080  796073  A  ADD   499  0.5384  0.5964  0.1673   
4   1  1:797392:A:G:rs541848210  797392  A  ADD   499  0.7499  0.6438  0.2123   
6   1    1:889018:C:A:rs7538305  889018  C  ADD   499  0.9068  0.2501  0.5554   

      U95  ... A2 CHROM       POS           ID  REF ALT IMPUTED TYPED  \
0  0.9018  ...  G     1  769257.0  rs142559957    G   A     1.0   0.0   
1   57.54  ...  T     1  773755.0  rs528768073    T   C     1.0   0.0   
3   1.733  ...  G     1  796073.0  rs186002080    G   A     1.0   0.0   
4   2.649  ...  G     1  797392.0  rs541848210    G   A     1.0   0.0   
6   1.481  ...  A     1  889018.0    rs7538305    A   C     1.0   0.0   

        MAF        R2  
0  0.059302  0.314921  
1  0.018825  0.321855  
3 

In [17]:
merged_df = merged_df.sort_values(by=["CHR", "BP"])
merged_df = merged_df.rename(columns={"NMISS": "N", "R2":"INFO"})
merged_df = merged_df[["CHR", "BP", "SNP", "A1", "A2", "N", "SE", "P", "OR", "INFO", "MAF"]]
print(merged_df.head())

  CHR      BP                       SNP A1 A2    N      SE        P      OR  \
0   1  769257  1:769257:A:G:rs142559957  A  G  499  0.3847  0.02583  0.4243   
1   1  773755  1:773755:C:T:rs528768073  C  T  499   1.063  0.06421   7.158   
2   1  794299    1:794299:C:G:rs4951859  C  G  499    0.18   0.7813  0.9513   
3   1  796073  1:796073:A:G:rs186002080  A  G  499  0.5964   0.2992  0.5384   
4   1  797392  1:797392:A:G:rs541848210  A  G  499  0.6438   0.6549  0.7499   

       INFO       MAF  
0  0.314921  0.059302  
1  0.321855  0.018825  
2  0.322603  0.263560  
3  0.322947  0.020123  
4  0.348903  0.016065  


In [20]:
merged_df.to_csv('/scratch/eleiterw/AppliedProject/annotatedBCF/imputationAndSummaryStats.txt.gz', sep='\t', compression="gzip", index=False)